# Cutout visualization and FITS export

Demonstrates:
1. Matplotlib quicklook of a single cutout
2. Astropy `ZScaleInterval` / `AsinhStretch` for science-grade display
3. Batch visualization of a tile
4. Napari 3-D stack viewer (optional)
5. Export to FITS for DS9 / Aladin

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from data_lake.io.cutouts import CutoutAccessor
from data_lake.export.to_fits import export_cutout, verify_round_trip

LAKE_ROOT = Path("/data/lake")
SURVEY = "des_dr2"
EXAMPLE_SOURCE_ID = 12345678  # replace with a real source_id

## 1. Load a single cutout

In [ ]:
acc = CutoutAccessor(LAKE_ROOT, SURVEY)
image, wcs = acc.get_cutout(EXAMPLE_SOURCE_ID)

print(f"Image shape (B, H, W): {image.shape}")
print(f"WCS centre (RA, Dec): {wcs.crval}")
print(f"Pixel scale: {abs(wcs.cd_matrix[0,0]) * 3600:.3f} arcsec/px")

## 2. Matplotlib quicklook

In [ ]:
def show_cutout(image: np.ndarray, title: str = "", band_labels: list[str] | None = None) -> None:
    """Show all bands of a cutout side by side."""
    n_bands = image.shape[0]
    fig, axes = plt.subplots(1, n_bands, figsize=(3 * n_bands, 3))
    if n_bands == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        band = image[i]
        vmin, vmax = np.nanpercentile(band, [1, 99])
        ax.imshow(band, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
        ax.set_title(band_labels[i] if band_labels else f"Band {i}")
        ax.axis("off")
    if title:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()


band_labels = acc.info.get("band_names") or None
show_cutout(image, title=f"source_id={EXAMPLE_SOURCE_ID}", band_labels=band_labels)

## 3. Astropy ZScale + AsinhStretch (science-grade display)

In [ ]:
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize

n_bands = image.shape[0]
fig, axes = plt.subplots(1, n_bands, figsize=(3 * n_bands, 3))
if n_bands == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    band = image[i]
    norm = ImageNormalize(band, interval=ZScaleInterval(), stretch=AsinhStretch())
    ax.imshow(band, origin="lower", cmap="gray", norm=norm)
    ax.set_title(band_labels[i] if band_labels else f"Band {i}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. RGB composite (requires at least 3 bands)

In [ ]:
def make_rgb(
    image: np.ndarray,
    r_idx: int = 2,  # adjust to your band ordering
    g_idx: int = 1,
    b_idx: int = 0,
    q: float = 8.0,
    scale: float = 0.02,
) -> np.ndarray:
    """Lupton-style RGB composite using asinh scaling."""
    r = image[r_idx].clip(0)
    g = image[g_idx].clip(0)
    b = image[b_idx].clip(0)
    i = (r + g + b) / 3.0 + 1e-10
    factor = np.arcsinh(q * scale * i) / (q * i)
    rgb = np.stack([r * factor, g * factor, b * factor], axis=-1)
    rgb = rgb / rgb.max()
    return np.clip(rgb, 0, 1)


if image.shape[0] >= 3:
    rgb = make_rgb(image)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(rgb, origin="lower")
    ax.set_title(f"RGB composite – source_id={EXAMPLE_SOURCE_ID}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("RGB composite requires at least 3 bands.")

## 5. Batch viewer – browse a tile

In [ ]:
TILE_NPIX = 1234  # replace with an actual tile pixel in your lake
N_SHOW = 16       # show this many cutouts from the tile

source_ids_in_tile = []
images_in_tile = []

for sid, img, _wcs in acc.iter_tile(TILE_NPIX):
    source_ids_in_tile.append(sid)
    images_in_tile.append(img)
    if len(images_in_tile) >= N_SHOW:
        break

print(f"Loaded {len(images_in_tile)} cutouts from tile {TILE_NPIX}")

# Show first band of each cutout in a grid
cols = 8
rows = (len(images_in_tile) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2 * rows))
for ax in axes.flat:
    ax.axis("off")

for ax, sid, img in zip(axes.flat, source_ids_in_tile, images_in_tile):
    band = img[0]
    vmin, vmax = np.nanpercentile(band, [1, 99])
    ax.imshow(band, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(str(sid), fontsize=6)
    ax.axis("off")

plt.suptitle(f"Tile Npix={TILE_NPIX} – first {N_SHOW} sources", y=1.01)
plt.tight_layout()
plt.show()

## 6. Napari 3-D stack viewer (optional)

Install napari with `pip install napari[all]`, then uncomment the cell below.

In [ ]:
# import napari
#
# stack = np.stack(images_in_tile)  # shape (N, B, H, W)
# viewer = napari.Viewer()
# for b in range(stack.shape[1]):
#     label = band_labels[b] if band_labels else f"band_{b}"
#     viewer.add_image(stack[:, b, :, :], name=label, colormap="gray")
# napari.run()

## 7. Export to FITS (for DS9 / Aladin)

In [ ]:
fits_path = export_cutout(
    lake_root=LAKE_ROOT,
    survey=SURVEY,
    source_id=EXAMPLE_SOURCE_ID,
    output_path=f"/tmp/cutout_{EXAMPLE_SOURCE_ID}.fits",
    accessor=acc,
)
print(f"Exported: {fits_path}")

## 8. Verify round-trip (pixel values + WCS)

In [ ]:
ok = verify_round_trip(LAKE_ROOT, SURVEY, EXAMPLE_SOURCE_ID)
print("Round-trip OK" if ok else "Round-trip FAILED")

## 9. WCS-projected display with astropy

In [ ]:
astropy_wcs = wcs.to_astropy()

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection=astropy_wcs)

band = image[0]
norm = ImageNormalize(band, interval=ZScaleInterval(), stretch=AsinhStretch())
ax.imshow(band, origin="lower", cmap="gray", norm=norm)

ax.coords[0].set_axislabel("RA (J2000)")
ax.coords[1].set_axislabel("Dec (J2000)")
ax.set_title(f"source_id={EXAMPLE_SOURCE_ID} – WCS projection")
plt.tight_layout()
plt.show()